In [1]:
# 3.1 Import Library
import pandas as pd
import numpy as np
import os
import ast
import json
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

In [2]:
# 3.2 Konfigurasi Path
LEXICON_PATH = '../../../kamus/sla_lexicon_adapted.txt' 
INPUT_FILE   = '../../outputs/SLA/sla_lexicon_matching_tanpa_perlakuan.csv'
OUTPUT_DIR   = '../../outputs/SLA'
OUTPUT_FILE  = os.path.join(OUTPUT_DIR, 'sla_vader_heuristics_tanpa_perlakuan.csv')
STATS_FILE   = os.path.join(OUTPUT_DIR, 'sla_vader_stats.json')
CLEAN_OUTPUT_FILE = os.path.join(OUTPUT_DIR, 'sla_sentiment_clean_tanpa_perlakuan.csv')

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [3]:
# 3.3 Load Data dan Lexicon
def load_custom_lexicon(filepath):
    lexicon = {}
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) >= 2:
                word = parts[0]
                mean_score = float(parts[1])
                lexicon[word] = mean_score
    return lexicon

# Load Lexicon
print(f"[INFO] Memuat leksikon adaptasi dari: {LEXICON_PATH}")
sla_dict = load_custom_lexicon(LEXICON_PATH)
print(f"[INFO] Berhasil memuat {len(sla_dict)} entri leksikon.")

# Load Data Matching
df = pd.read_csv(INPUT_FILE)

# Convert kolom list string -> list python (sesuai output matching no_function)
list_cols = ['tokens', 'matched_words', 'unmatched_words']
for col in list_cols:
    if col in df.columns:
        df[col] = df[col].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

# ==========================================
# PERUBAHAN UTAMA: TANPA IGNORE / REMOVE
# ==========================================
# Rekonstruksi sequence langsung dari tokens asli menggunakan skor native leksikon
def reconstruct_sequence_full(tokens, lexicon):
    seq_tokens = []
    seq_scores = []
    for tok in tokens:
        t_low = tok.lower()
        seq_tokens.append(tok)
        if t_low in lexicon:
            # Ambil skor asli dari leksikon
            seq_scores.append({'word': tok, 'score': lexicon[t_low], 'matched': True})
        else:
            # Jika tidak ada di leksikon, skor 0.0
            seq_scores.append({'word': tok, 'score': 0.0, 'matched': False})
    return seq_tokens, seq_scores

# Apply ke dataframe
df[['filtered_tokens', 'filtered_word_scores']] = df.apply(
    lambda row: pd.Series(reconstruct_sequence_full(row['tokens'], sla_dict)),
    axis=1
)

print(f"[INFO] Data loaded: {len(df)} tweets")
print("[INFO] No Function Filter Applied (Full Sequence & Native Scores).")

[INFO] Memuat leksikon adaptasi dari: ../../../kamus/sla_lexicon_adapted.txt
[INFO] Berhasil memuat 9855 entri leksikon.
[INFO] Data loaded: 450 tweets
[INFO] No Function Filter Applied (Full Sequence & Native Scores).


In [4]:
# 3.4 Parameter Heuristik VADER (REVISI)

# Punctuation
PUNCT_BOOST = 0.292
MAX_EXCLAMATION = 4

# Capitalization
CAP_BOOST = 0.733

# Booster / Dampener
B_INCR = 0.293
B_DECR = -0.293

BOOSTERS = {
    'sangat': B_INCR,
    'sekali': B_INCR,
    'banget': B_INCR,
    'amat': B_INCR,
    'teramat': B_INCR,
    'terlalu': B_INCR,
    'benar': B_INCR,
    'betul': B_INCR,
    'pasti': B_INCR,
    'jelas': B_INCR,
    'nyata': B_INCR,
    'parah': B_INCR,

    'agak': B_DECR,
    'sedikit': B_DECR,
    'kurang': B_DECR,
    'hampir': B_DECR,
    'nyaris': B_DECR,
    'hanya': B_DECR,
    'cuma': B_DECR,
    'sekadar': B_DECR,
    'mungkin': B_DECR,
    'lumayan': B_DECR,
    'cukup': B_DECR
}

# Contrastive conjunction
CONTRAST_CONJUNCTIONS = [
    'tapi', 'tetapi', 'namun',
    'walaupun', 'meskipun',
    'kendati', 'sedangkan',
    'padahal'
]

# Negation
NEGATIONS = [
    'tidak', 'tak', 'nggak',
    'gak', 'bukan', 'jangan',
    'tanpa', 'belum', 'never',
    'no'
]

NEGATION_WINDOW = 3
NEGATION_SCALAR = -0.74

# Normalization
ALPHA = 15.0

print("[INFO] Heuristic Parameters loaded.")

[INFO] Heuristic Parameters loaded.


In [5]:
# 3.5 Fungsi Heuristik VADER (FULL REVISI)
# Parameter Utama
PUNCT_BOOST     = 0.292
MAX_PUNCT_BOOST = 1.168
CAP_BOOST       = 0.733

B_INCR = 0.293
B_DECR = -0.293

NEGATION_SCALAR = -0.74
NEGATION_WINDOW = 3

ALPHA = 15.0


# Booster / Degree Modifiers
BOOSTERS = {
    'sangat': B_INCR,
    'sekali': B_INCR,
    'banget': B_INCR,
    'amat': B_INCR,
    'teramat': B_INCR,
    'terlalu': B_INCR,
    'benar': B_INCR,
    'betul': B_INCR,
    'pasti': B_INCR,
    'jelas': B_INCR,
    'nyata': B_INCR,
    'parah': B_INCR,
    'luar': B_INCR,
    'luar biasa': B_INCR,

    'agak': B_DECR,
    'sedikit': B_DECR,
    'kurang': B_DECR,
    'hampir': B_DECR,
    'nyaris': B_DECR,
    'hanya': B_DECR,
    'cuma': B_DECR,
    'sekadar': B_DECR,
    'mungkin': B_DECR,
    'lumayan': B_DECR,
    'cukup': B_DECR
}


# Contrastive Conjunctions
CONTRAST_CONJUNCTIONS = [
    'tapi', 'tetapi', 'namun', 'walaupun',
    'meskipun', 'kendati', 'sekalipun',
    'sedangkan', 'padahal', 'sementara',
    'biarpun', 'walau', 'meski'
]


# Negation Words
NEGATIONS = [
    'tidak', 'tak', 'nggak', 'gak',
    'bukan', 'jangan', 'tanpa',
    'belum', 'no', 'never'
]

print("[INFO] Heuristic parameters loaded.")

[INFO] Heuristic parameters loaded.


In [6]:
# 3.5.1 Punctuation Heuristic

def apply_punctuation_heuristic(text, base_score):

    if not isinstance(text, str):
        return base_score, {
            'exclamation_count': 0,
            'question_count': 0,
            'punct_boost': 0
        }

    exclamation_count = text.count('!')
    question_count = text.count('?')

    # Exclamation emphasis
    ep_amplifier = min(exclamation_count, 4) * 0.292

    # Question mark emphasis
    qm_amplifier = 0

    if question_count > 1:
        if question_count <= 3:
            qm_amplifier = question_count * 0.18
        else:
            qm_amplifier = 0.96

    punct_boost = ep_amplifier + qm_amplifier

    if base_score > 0:
        final_score = base_score + punct_boost
    elif base_score < 0:
        final_score = base_score - punct_boost
    else:
        final_score = base_score

    return final_score, {
        'exclamation_count': exclamation_count,
        'question_count': question_count,
        'punct_boost': punct_boost
    }

print("[INFO] Punctuation heuristic ready.")

[INFO] Punctuation heuristic ready.


In [7]:
# 3.5.2 Capitalization Heuristic

def apply_capitalization_heuristic(tokens, word_scores):

    sentiments = []

    allcaps_words = [t for t in tokens if t.isupper()]
    is_cap_diff = 0 < len(allcaps_words) < len(tokens)

    for i, token_data in enumerate(word_scores):

        score = token_data['score']
        token = tokens[i]

        if score != 0 and token.isupper() and is_cap_diff:

            if score > 0:
                score += CAP_BOOST
            else:
                score -= CAP_BOOST

        sentiments.append(score)

    adjusted_score = sum(sentiments)

    return adjusted_score, {
        'cap_diff': is_cap_diff,
        'caps_words': len(allcaps_words)
    }

print("[INFO] Capitalization heuristic ready.")

[INFO] Capitalization heuristic ready.


In [8]:
# 3.5.3 Degree Modifier Heuristic

def apply_degree_modifier_heuristic(tokens, word_scores):

    adjusted_scores = []
    adjustments = []

    for i, token_data in enumerate(word_scores):

        score = token_data['score']

        if score == 0:
            adjusted_scores.append(score)
            continue

        modified_score = score

        # Cek 3 token sebelumnya
        for distance in range(1, 4):

            prev_idx = i - distance

            if prev_idx < 0:
                continue

            prev_token = tokens[prev_idx].lower()

            if prev_token in BOOSTERS:

                scalar = BOOSTERS[prev_token]

                # Polarity-sensitive
                if modified_score < 0:
                    scalar *= -1

                # Distance decay
                if distance == 2:
                    scalar *= 0.95
                elif distance == 3:
                    scalar *= 0.9

                modified_score += scalar

                adjustments.append({
                    'modifier': prev_token,
                    'affected_word': tokens[i],
                    'scalar': scalar
                })

        adjusted_scores.append(modified_score)

    return sum(adjusted_scores), adjustments

print("[INFO] Degree modifier heuristic ready.")

[INFO] Degree modifier heuristic ready.


In [9]:
# 3.5.4 Negation Heuristic

def apply_negation_heuristic(tokens, word_scores):

    adjusted_scores = []
    negations_found = []

    for i, token_data in enumerate(word_scores):

        score = token_data['score']

        if score == 0:
            adjusted_scores.append(score)
            continue

        modified_score = score

        # Cek 3 token sebelumnya
        for distance in range(1, NEGATION_WINDOW + 1):

            prev_idx = i - distance

            if prev_idx < 0:
                continue

            prev_token = tokens[prev_idx].lower()

            if prev_token in NEGATIONS:

                modified_score *= NEGATION_SCALAR

                negations_found.append({
                    'negation': prev_token,
                    'affected_word': tokens[i],
                    'distance': distance
                })

                break

        adjusted_scores.append(modified_score)

    return sum(adjusted_scores), negations_found

print("[INFO] Negation heuristic ready.")

[INFO] Negation heuristic ready.


In [10]:
# 3.5.5 Contrastive Conjunction Heuristic

def apply_polarity_shift_heuristic(tokens, word_scores):

    sentiments = [w['score'] for w in word_scores]
    lowered_tokens = [t.lower() for t in tokens]

    contrast_positions = [
        i for i, t in enumerate(lowered_tokens)
        if t in CONTRAST_CONJUNCTIONS
    ]

    if not contrast_positions:
        return sum(sentiments), {
            'contrast_found': False
        }

    but_idx = contrast_positions[0]

    adjusted_sentiments = []

    for i, score in enumerate(sentiments):

        if i < but_idx:
            adjusted_sentiments.append(score * 0.5)

        elif i > but_idx:
            adjusted_sentiments.append(score * 1.5)

        else:
            adjusted_sentiments.append(score)

    return sum(adjusted_sentiments), {
        'contrast_found': True,
        'contrast_word': tokens[but_idx],
        'contrast_position': but_idx
    }

print("[INFO] Contrastive conjunction heuristic ready.")

[INFO] Contrastive conjunction heuristic ready.


In [11]:
# 3.6 Implementasi Scoring (Sequential Pipeline)
print("\n[PROSES] Menjalankan Scoring SLA + Heuristics (No Function)...")
results = []

for idx, row in df.iterrows():
    if (idx + 1) % 1000 == 0:
        print(f"Processing tweet {idx + 1}/{len(df)}")

    text = row['teks']
    tokens = row['filtered_tokens']
    word_scores = row['filtered_word_scores']
    
    current_scores = [w['score'] for w in word_scores]
    
    # 1. Degree Modifiers
    modified_scores = current_scores.copy()
    for i in range(len(modified_scores)):
        if modified_scores[i] == 0: continue
        for dist in range(1, 4):
            prev_idx = i - dist
            if prev_idx < 0: continue
            prev_tok = tokens[prev_idx].lower()
            if prev_tok in BOOSTERS:
                scalar = BOOSTERS[prev_tok]
                if modified_scores[i] < 0: scalar *= -1
                if dist == 2: scalar *= 0.95
                elif dist == 3: scalar *= 0.90
                modified_scores[i] += scalar

    # 2. Negation Scaling
    negated_scores = modified_scores.copy()
    for i in range(len(negated_scores)):
        if negated_scores[i] == 0: continue
        for dist in range(1, NEGATION_WINDOW + 1):
            prev_idx = i - dist
            if prev_idx < 0: continue
            prev_tok = tokens[prev_idx].lower()
            if prev_tok in NEGATIONS:
                negated_scores[i] *= NEGATION_SCALAR
                break

    # 3. Contrastive Conjunction Shift
    lowered_tokens = [t.lower() for t in tokens]
    contrast_idx = next((i for i, t in enumerate(lowered_tokens) if t in CONTRAST_CONJUNCTIONS), None)
    contrast_scores = negated_scores.copy()
    if contrast_idx is not None:
        contrast_scores = [
            s * 0.5 if i < contrast_idx else s * 1.5 if i > contrast_idx else s 
            for i, s in enumerate(contrast_scores)
        ]

    # 4. Sum & Heuristics
    total_score = sum(contrast_scores)
    punct_adjusted, _ = apply_punctuation_heuristic(text, total_score)
    
    all_caps_ratio = sum(1 for t in tokens if t.isupper() and t.isalpha()) / max(len(tokens), 1)
    if 0.15 <= all_caps_ratio <= 0.85 and total_score != 0:
        cap_adjusted = punct_adjusted + (CAP_BOOST if total_score > 0 else -CAP_BOOST)
    else:
        cap_adjusted = punct_adjusted

    # 5. Normalization & Classification
    compound_score = cap_adjusted / np.sqrt(cap_adjusted**2 + ALPHA)
    compound_score = float(np.clip(compound_score, -1.0, 1.0))

    if compound_score >= 0.05: sentiment = 'positive'
    elif compound_score <= -0.05: sentiment = 'negative'
    else: sentiment = 'neutral'

    results.append({
        'no': row['no'],
        'timestamp': row.get('timestamp', ''),
        'teks': text,
        'teks_processed': row.get('teks_processed', ''),
        'base_score': total_score,
        'final_raw_score': cap_adjusted,
        'compound_score': compound_score,
        'sentiment': sentiment
    })

print(f"\n[SELESAI] Memproses {len(results)} tweet.")


[PROSES] Menjalankan Scoring SLA + Heuristics (No Function)...

[SELESAI] Memproses 450 tweet.


In [12]:
# 3.7 Statistik
df_results = pd.DataFrame(results)

stats = {
    'total_tweets': len(df_results),
    'sentiment_distribution': df_results['sentiment'].value_counts().to_dict(),
    'compound_score_stats': {
        'mean': float(df_results['compound_score'].mean()),
        'std': float(df_results['compound_score'].std()),
        'min': float(df_results['compound_score'].min()),
        'max': float(df_results['compound_score'].max())
    }
}

print("\n[STATISTIK] Distribusi Sentimen:")
for sent, count in stats['sentiment_distribution'].items():
    print(f"  {sent}: {count} ({count/len(df_results)*100:.2f}%)")

print("\n[PREVIEW] 5 Data Pertama:")
display(df_results[['no', 'teks_processed', 'compound_score', 'sentiment']].head())


[STATISTIK] Distribusi Sentimen:
  positive: 213 (47.33%)
  negative: 200 (44.44%)
  neutral: 37 (8.22%)

[PREVIEW] 5 Data Pertama:


,no,teks_processed,compound_score,sentiment
0,50,komisi IX ketua komisi IX DPR RI rekomendasi k...,-0.853113,negative
1,84,dapilku rumah reses dan silaturahim sama anggo...,0.773915,positive
2,89,haru wakil rakyat terlalu banyak waktu untuk u...,-0.933788,negative
3,101,ketua dan para wakil ketua DPR RI PERLU RUU LA...,0.898624,positive
4,121,viva anggota DPR minta TNI AU investigasi cela...,-0.819240,negative


In [13]:
# 3.8 Simpan Hasil ke CSV & JSON
df.to_csv(OUTPUT_FILE, index=False)
print(f"[INFO] Hasil utama disimpan: {OUTPUT_FILE}")

with open(STATS_FILE, 'w', encoding='utf-8') as f:
    json.dump(stats, f, indent=2, ensure_ascii=False)
print(f"[INFO] Statistik disimpan: {STATS_FILE}")

[INFO] Hasil utama disimpan: ../../outputs/SLA\sla_vader_heuristics_tanpa_perlakuan.csv
[INFO] Statistik disimpan: ../../outputs/SLA\sla_vader_stats.json


In [14]:
# 3.9 Simpan Versi Ringkas untuk Evaluasi (Notebook 04)
df_clean = df_results[['no', 'timestamp', 'teks', 'teks_processed', 'sentiment', 'compound_score']]
df_clean.to_csv(CLEAN_OUTPUT_FILE, index=False, encoding='utf-8')

print(f"\n[INFO] Hasil tersimpan di:")
print(f"  - Full   : {OUTPUT_FILE}")
print(f"  - Stats  : {STATS_FILE}")
print(f"  - Clean  : {CLEAN_OUTPUT_FILE} (Siap untuk evaluasi)")


[INFO] Hasil tersimpan di:
  - Full   : ../../outputs/SLA\sla_vader_heuristics_tanpa_perlakuan.csv
  - Stats  : ../../outputs/SLA\sla_vader_stats.json
  - Clean  : ../../outputs/SLA\sla_sentiment_clean_tanpa_perlakuan.csv (Siap untuk evaluasi)
